# Ebaluazioa 3.2: Klasifikazio Probabilistikoa (CIFAR-10) - Erantzun Ezezagunei Aurre Eginez
Helburua modeloei erantzuteko eskatzea denez aurrez _inoiz_ ikusi ez duten klase baten aurrean, entropia erabiliko da ziurgabetasuna gisa.

CIFAR-10 datu-sortarekin lan egingo dugu (irudiak koloretan, 32x32). Adibide honetan, **3 klasea ('Katua') ezkutatuko dugu** trebakuntzatik.
Hemen `CNNMCDropoutClassifier` berria erabiliko dugu, irudi konplexuagoetarako bereziki diseinatua dagoena.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.generators import CIFAR10Generator
from src.shadows import ClassShadow
from src.classifiers import RandomForestClassifierModel
from src.classifiers import CNNMCDropoutClassifier
from src.splitter import GapSplitter
from src.orchestrator import ClassificationOrchestrator
from src.visualization import plot_classification_results, plot_image_sample_uncertainties

In [ ]:
N_SAMPLES = 5000
shadow = ClassShadow(shadow_class=3)
noise_strategy = None

models = [
    RandomForestClassifierModel(n_estimators=50, max_depth=10),
    CNNMCDropoutClassifier(image_shape=(3, 32, 32), epochs=30, dropout_rate=0.3, batch_size=64, n_iter=50)
]

splitter = GapSplitter(test_size=0.2)
generator = CIFAR10Generator(train=True)

In [ ]:
for m in models:
    print(f"\n{'='*60}")
    print(f"Ebaluatzen: {m.get_name()}")
    print(f"{'='*60}")
    
    orchestrator = ClassificationOrchestrator(generator, splitter, m)
    
    artifacts = orchestrator.run(
        n_samples=N_SAMPLES,
        x_range=(0,0), 
        noise_strategy=noise_strategy,
        shadow=shadow
    )

    if artifacts.uncertainties is not None:
        ent_vis = np.mean(artifacts.uncertainties['visible'])
        ent_sha = np.mean(artifacts.uncertainties['shadow'])
        print(f"    * Zehaztasuna (Ikusgai): {artifacts.metrics.acc_test_visible:.4f}")
        print(f"    * Zehaztasuna (Itzala) : {artifacts.metrics.acc_test_shadow:.4f}")
        print(f"    * Entropia (Ikusgai)   : {ent_vis:.4f}")
        print(f"    * Entropia (Itzala)    : {ent_sha:.4f}")

    plot_classification_results(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, title=f"Eredua: {m.get_name()}")
    plot_image_sample_uncertainties(artifacts.split_data, artifacts.predictions, artifacts.uncertainties, num_samples=5)
    
print("\nEsperimentua Arrakastaz Bideratua")